# Introduction
In the world of hospitality, customer reviews are a goldmine of insights — but with thousands of reviews scattered across text, voice, video, and images, it's hard for businesses to make sense of them all.

An intelligent pipeline that takes in text reviews and outputs a structured, grounded summary — powered by RAG, embeddings, one shot prompting and text understanding.

This project builds a Smart Review Summarizer that:

# Phase 1 (By April 20th ) - Gen AI capabilities incorporated
1. Structured output/JSON model: Current program is storing the JSON structure in a output file.
2. Document understanding : The program currently reads the reviews from the sample data created in the the excel file.
3. Embeddings : Sentence transformers are used in this project - all-miniLM-L6-v2
4. Retrieval augmented genration (RAG) : Using FLAN-T5 model for genrating relevant summaries by retriving passges from the vector store based on
   the query
5. Vector store : FAISS index6. 
6. Classification of reviews based on
        cleanliness, location, value, service,food_availability,
        crowd_level, pricing,queue_fairness ,park_experience,
        emotional_impact , attractions , overall_experience

# Phase 2 (After Apr 20th)
1. Using agents to interact with live reviews using Google API
2. Using Context Cache
3. MLOps : Making the project production mode


# GitHub

https://github.com/prakashpillai/LearningGenAI.git




# Use Case
Imagine a hotel manager trying to understand guest feedback from Booking.com, Google reviews, and Trip advisor messages. Instead of reading hundreds of reviews manually, our system:

Summarizes guest experiences (cleanliness, location, service, value, etc.)
Categorizes pros/cons from real reviews
Grounds summaries in similar past reviews using RAG





# Step 1 : Install required libraries

In [ ]:
# Install the required libraries for embedding text,Hugging face core library
!pip install -q sentence-transformers transformers faiss-cpu scikit-learn pandas

# Step 2 : Importing the required libraries

In [ ]:
import json
import re
import csv
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from transformers import T5Tokenizer, T5ForConditionalGeneration
# for fast vector similarity search
import faiss

In [ ]:
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

candidate_labels = [
    "cleanliness", "location", "value", "service", "food_availability",
    "crowd_level", "pricing", "queue_fairness", "park_experience",
    "emotional_impact", "attractions", "overall_experience"
]

### Step 3 : Load a sample Dataset from Excel
Creating a simple dataset of reviews using Excel and uploaded the file in the dataset.Reading the excel using Pandas.

In [ ]:
# Load the Excel file containing user reviews
# Remove any rows with missing review text to ensure clean input
# Extract reviews as a list
#loading sample data from file in the pandas dataframe.
df = pd.read_excel('/kaggle/input/test-data/Dataset_Kaggle_project.xlsx', engine='openpyxl')
#Removes rows with missing values
df = df.dropna(subset=["Review"])
texts = df['Review'].tolist()
df.head()


# Step 4: Load the pre-trained model

In [ ]:
# Load SentenceTransformer model for embeddings
# Load FLAN-T5 for prompt-based text generation (cluster labeling)
model_name = 'google/flan-t5-large'
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

Generate text embeddings for each review using pre-trained model and store them in a FAISS vector database.

# Step 5 # Create the embeddings and index for similarity search (FAISS index)

In [ ]:
embeddings = embed_model.encode(texts, show_progress_bar=True)
dimension = embeddings.shape[1]
# Build a FAISS index for fast similarity querying
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Step 6 : Clause level categorization - Text split and preprocessing

In [ ]:
# Utility to break down long reviews into shorter, semantically meaningful clauses
# Helps improve clustering and labeling accuracy
def split_review_into_clauses(review):
    raw_clauses = re.split(r'[.!?;]+', review)
    return [c.strip() for c in raw_clauses if len(c.strip().split()) >= 5]

# Step 7: Clause Categorization and sentiment detection

In [ ]:
# Load external rule files (uploaded to Kaggle)
with open("/kaggle/input/sentiment-overrides/category_rules.json") as f:
    CATEGORY_RULES = json.load(f)
with open("/kaggle/input/sentiment-overrides/sentiment_overrides.json") as f:
    SENTIMENT_OVERRIDES = json.load(f)

# Load sentiment model
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
CONFIDENCE_THRESHOLD = 0.6
DEBUG = False  # Toggle debug prints

# Match keywords inside a clause
def match_keywords(clause, keywords):
    return any(k in clause for k in keywords)

def analyze_clause(clause):
    clause = clause.strip().lower()
    entry = {"text": clause, "category_sentiments": {}}

     # Match categories
    for category, keywords in CATEGORY_RULES.items():
        if match_keywords(clause, keywords):
            entry["category_sentiments"][category] = None  # sentiment to be added

    # Sentiment override
    overridden = None
    for label, phrases in SENTIMENT_OVERRIDES.items():
        if match_keywords(clause, phrases):
            overridden = label
            break

    # If no categories matched, assign 'uncategorized'
    if not entry["category_sentiments"]:
        entry["category_sentiments"]["uncategorized"] = None

    # Run sentiment model
    try:
        result = sentiment_model(clause)[0]
        model_label = result["label"].lower()
        score = result["score"]

        sentiment = (
            "neutral" if score < CONFIDENCE_THRESHOLD
            else "positive" if "positive" in model_label
            else "negative" if "negative" in model_label
            else "neutral"
        )
    except Exception as e:
        print(f"Sentiment analysis failed for: {clause}\nError: {e}")
        sentiment = "neutral"

    # Assign sentiment per category
    final_sentiment = overridden if overridden else sentiment
    for cat in entry["category_sentiments"].keys():
        entry["category_sentiments"][cat] = final_sentiment

    return entry


# Categorize a list of reviews
def categorize_reviews_multiclause(review_list):
    categorized = []
    for review in review_list:
        clauses = split_review_into_clauses(review)
        for clause in clauses:
            if clause.strip():
                result = analyze_clause(clause)
                if DEBUG:
                    cat_sent = ", ".join([f"{k}: {v}" for k, v in result["category_sentiments"].items()])
                    print(f"DEBUG → Clause: {clause} → {cat_sent}")
                categorized.append(result)
    return categorized

# Step 8: Cluster the categorized clauses

In [ ]:
# Using KMeans to group similar review clauses together based on semantic embeddings
def cluster_clauses(categorized_reviews, n_clusters=8):
    clause_texts = [entry["text"] for entry in categorized_reviews]
    clause_embeddings = embed_model.encode(clause_texts, show_progress_bar=True)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(clause_embeddings)

    clusters = {}
    for i, label in enumerate(labels):
        clusters.setdefault(label, []).append(clause_texts[i])
    
    return clusters

# Step 9: Define Cluster label to Category mapping

In [ ]:
# This dictionary maps generated FLAN-T5 cluster labels to standardized categories
cluster_label_to_category = {
    "wait time": "wait_times",
    "hot dogs": "food_availability",
    "great hotel in a great location": "hotel_quality",
    "the disneyland parks": "park_experience",
    "the line was longer than the ride's lines": "queue_fairness",
    "the price is high": "pricing",
    "love this place always when we are there is magical": "magical_experience",
    "a mediocre experience": "overall_experience",
    "a lot of rides": "attractions",
    "the best hotel in vegas": "emotional_impact",
}

# Step 10 : Label clusters with FLAN-T5

In [ ]:
# For each cluster, FLAN-T5 generates a human-readable label summarizing the common theme
def label_clusters_with_flan(clusters, tokenizer, model, max_examples_per_cluster=5):
    labeled_clusters = {}

    for cluster_id, samples in clusters.items():
        if len(samples) < 2:
            print(f"\n⏭️ Skipping Cluster {cluster_id} (not enough samples)")
            continue

        prompt = "Given the following hotel review statements, suggest a short topic label (1–3 words):\n\n"
        for i, clause in enumerate(samples[:max_examples_per_cluster]):
            prompt += f"{i+1}. {clause.strip()}\n"
        prompt += "\nLabel:"

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
        outputs = model.generate(**inputs, max_new_tokens=10)
        label = tokenizer.decode(outputs[0], skip_special_tokens=True).split("\n")[0].split(".")[0].strip().lower()
        category = cluster_label_to_category.get(label, "uncategorized")

        labeled_clusters[label] = {"category": category, "clauses": samples}
        print(f"\n🏷️ Cluster {cluster_id} → {label} → Category: {category}")
        for s in samples:
            print(f" - {s}")
    
    return labeled_clusters

# Step 11 : Assign cluster Categories to reivews

In [ ]:
def assign_cluster_categories_to_reviews(categorized_reviews, labeled_clusters):
    # Build lookup: clause text (lowercased) → inferred category (cluster label)
    clause_to_category = {
        text.strip().lower(): label
        for label, cluster in labeled_clusters.items()
        for text in cluster
    }

    for entry in categorized_reviews:
        clause_text = entry["text"].strip().lower()
        inferred_cat = clause_to_category.get(clause_text)

        # Initialize if not present
        if "category_sentiments" not in entry:
            entry["category_sentiments"] = {}

        current_cats = entry["category_sentiments"].keys()

        # Only assign if it's completely uncategorized or only has 'uncategorized'
        if inferred_cat and (not current_cats or current_cats == {"uncategorized"}):
            sentiment = entry.get("sentiment", "neutral")
            entry["category_sentiments"] = {inferred_cat: sentiment}
            entry["categories"] = [inferred_cat]  # Optional for compatibility

    return categorized_reviews

# Genrate summary and Categorize Top reviews based on query

In [ ]:
def debug_grounded_summary_steps(
    query,
    top_k_reviews,
    categorized_reviews,
    category_counts,
    sentiment_counts,
    prompt,
    summary
):
    print("\n🔍 Step 1: Top-k Retrieved Reviews:")
    for i, review in enumerate(top_k_reviews, 1):
        print(f"{i}. {review}")

    print("\n🧩 Step 2: Categorized Clauses:")
    for item in categorized_reviews:
        print(f"Text: {item['text']}, Categories: {item['categories']}, Sentiment: {item['sentiment']}")

    print("\n📊 Step 3: Stats Summary:")
    print("Top Categories:", category_counts)
    print("Sentiment Counts:", sentiment_counts)

    print("\n📚 Step 4: Prompt Sent to FLAN:")
    print(prompt)

    print("\n📝 Step 4: Generated Summary:")
    print(summary)

In [ ]:
def grounded_summary(query, k=20, debug=False):
    from collections import Counter

    # Step 1: Retrieve top-k most semantically similar reviews
    query_embedding = embed_model.encode([query])
    D, I = index.search(query_embedding, k)
    top_k_reviews = [texts[i] for i in I[0]]

    if debug:
        print("🔍 Step 1: Top-k Retrieved Reviews:")
        for i, review in enumerate(top_k_reviews, 1):
            print(f"{i}. {review}")

    # Step 2: Categorize clauses
    categorized_reviews = categorize_reviews_multiclause(top_k_reviews)

    if debug:
        print("\n🧩 Step 2: Categorized Clauses:")
        for item in categorized_reviews:
            print(f"Text: {item['text']}, Category Sentiments: {item.get('category_sentiments', {})}")


    # Step 3: Stats
    all_categories = [
        cat.lower()
        for review in categorized_reviews
        for cat in review["category_sentiments"].keys()
        if cat.lower() != "uncategorized"
        ]
    
    all_sentiments = [
        sentiment
        for review in categorized_reviews
        for sentiment in review["category_sentiments"].values()
        if sentiment
    ]

    category_counts = Counter(all_categories)
    sentiment_counts = Counter(all_sentiments)

    total_cats = len(all_categories)
    top_categories = category_counts.most_common(3)
    top_categories_str = ", ".join([
        f"{cat} ({(count/total_cats)*100:.0f}%)" for cat, count in top_categories
    ])

    total_sentiments = len(all_sentiments)
    positive_pct = (sentiment_counts["positive"] / total_sentiments) * 100 if total_sentiments else 0
    negative_pct = (sentiment_counts["negative"] / total_sentiments) * 100 if total_sentiments else 0
    neutral_pct = (sentiment_counts["neutral"] / total_sentiments) * 100 if total_sentiments else 0

    top_sentiment = max(sentiment_counts, key=sentiment_counts.get) if sentiment_counts else "neutral"

    if debug:
        print("\n📊 Step 3: Stats Summary:")
        print(f"Top Categories: {top_categories_str}")
        print(f"Sentiment → Positive: {positive_pct:.0f}%, Neutral: {neutral_pct:.0f}%, Negative: {negative_pct:.0f}%")

    # Step 4: Prepare Summary Input
    # Prepare cleaner, focused input using only clause texts
    positive_clauses = [r["text"] for r in categorized_reviews if "positive" in r["category_sentiments"].values()]
    negative_clauses = [r["text"] for r in categorized_reviews if "negative" in r["category_sentiments"].values()]
    neutral_clauses = [r["text"] for r in categorized_reviews if "neutral" in r["category_sentiments"].values()]

    
    # Sample balance (optional: tune these numbers)
    clauses_to_use = positive_clauses[:10] + negative_clauses[:10] + neutral_clauses[:2]
    review_text = " ".join(clauses_to_use)
    
    # Optimized prompt
    prompt = (
        "You are summarizing visitor reviews about Disneyland Parks.\n"
        "Summarize what guests liked 👍, disliked 👎, and suggest improvements 🛠️.\n"
        "Use bullet points and avoid repetition.\n\n"
        "Examples:\n"
        "- 👍 Friendly staff, clean rooms, and great location.\n"
        "- 👎 Long wait times and high food prices.\n"
        "- 🛠️ Improve queue management and offer more food variety.\n\n"
        f"Guest comments:\n{review_text}\n\n"
        "Summary:"
    )




    if debug:
        print("\n🧠 Step 4: Prompt Sent to FLAN:")
        print(prompt)

    # Step 5: Generate Summary
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=200, num_beams=4, no_repeat_ngram_size=4, early_stopping=True)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Fallback if too short
    if len(summary.split()) < 10:
        summary += (
            "\n- 👎 Some guests noted long wait times and expensive food.\n"
            "- 🛠️ Improvements in food service speed and queue fairness recommended."
        )

    if debug:
        print("\n📝 Step 5: FLAN Summary:")
        print(summary)

    # Final Enriched Output
    enriched_summary = (
        f"{summary}\n\n"
        f"🗂️ Top categories: {top_categories_str}\n"
        f"😊 Sentiment breakdown → Positive: {positive_pct:.0f}%, Neutral: {neutral_pct:.0f}%, Negative: {negative_pct:.0f}%\n"
        f"💬 Overall sentiment: {top_sentiment}"
    )

    return {
        "query": query,
        "summary": enriched_summary,
        "top_k_reviews": top_k_reviews,
        "categorized_reviews": categorized_reviews
    }

# Step 12: Run the full pipeline

In [ ]:
# 1. Run the grounding/summarization module to extract relevant reviews
output = grounded_summary("Disneyland Parks", k=20)

# 2. Cluster the clauses from those reviews
clusters = cluster_clauses(output["categorized_reviews"], n_clusters=8)

# 3. Generate human-readable labels for the clusters
labeled_clusters = label_clusters_with_flan(clusters, tokenizer, model)

# 4. Assign those labels/categories back to the individual clause entries
output["categorized_reviews"] = assign_cluster_categories_to_reviews(
    output["categorized_reviews"], labeled_clusters
)

# 5. Save labeled clusters in the output for export
output["labeled_clusters"] = labeled_clusters


# Check the results of Summary

In [ ]:
output = grounded_summary("Disneyland Paris", k=10, debug=True)
print("🔍 Summary:")
print(output["summary"])

# Adding visualization to the review data based on JSON genrated.



# Phase 2 - Future product enchancments
1. To review the images.
2. Use Agents to interact with live reviews using API
3. Adding the MLOps to make the product